# 25 · Explicit Physical-State Gray-Box Layer — H1

This notebook implements the pre-declared Phase-H physical layer. It **does not change the frozen v3 operational policy**. It verifies the 16-feature SHARP contract, builds interpretable magnetic-state axes and temporal evolution, fits a low-capacity physical logistic reference using early training support only, fits a training-only physical-state applicability model, and exports diagnostics for later H2 policy development.

**Evidence boundary:** Cycle-25 and 2026 outcomes have already been seen in A–G. Any later-regime result produced here is post-hoc and cannot be presented as untouched confirmation.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

ROOT = Path("/home/abmoses2000")
REPO = Path.cwd()
if not (REPO/"results"/"dataset_package_20261002"/"manifest.json").exists():
    for p in [Path("/home/abmoses2000/gray-box-solar-flare-forecasting"),
              Path("/home/abmoses2000/gray_box_solar_flare_forecasting")]:
        if (p/"results"/"dataset_package_20261002"/"manifest.json").exists():
            REPO=p; break

MANIFEST = REPO/"results"/"dataset_package_20261002"/"manifest.json"
if not MANIFEST.exists():
    standalone_manifest = ROOT/"graybox_phase_h_dataset_manifest_20261002.json"
    if standalone_manifest.exists():
        MANIFEST = standalone_manifest

SHARP = ROOT/"graybox_phase_h_recovered_input_20261006"/"sharp.npy"
SHARP_LEDGER = ROOT/"sharp72_predictions_frozen_20261002.csv.gz"
MASTER = ROOT/"graybox_aia72_master_predictions_v2_20261005"/"graybox_aia72_master_predictions.csv.gz"
OUT = ROOT/"graybox_aia72_physical_state_h1_v1_20261006"

for p in [MANIFEST, SHARP, SHARP_LEDGER, MASTER]:
    assert p.is_file(), f"Missing required frozen input: {p}"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""): h.update(chunk)
    return h.hexdigest()

manifest=json.loads(MANIFEST.read_text())
EXPECTED_FEATURES=["MEANGBZ","MEANGAM","MEANGBT","MEANGBH","MEANJZD","TOTUSJZ","MEANALP","MEANJZH","ABSNJZH","SAVNCPP","MEANSHR","SHRGT45","R_VALUE","USFLUX","TOTPOT","TOTUSJH"]
assert manifest["feature_order"]==EXPECTED_FEATURES, "SHARP feature order changed"
assert manifest["sharp_shape"]==[113433,3,16], "SHARP shape contract changed"

SHARP_SHA="c9921be7b7fe381b43436a0e927d15d84de83e15376025ae0d8a79a6ef917639"
LEDGER_SHA="7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"
MASTER_SHA="17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

assert sha256(SHARP)==SHARP_SHA, "Recovered canonical sharp.npy hash mismatch"
assert sha256(SHARP_LEDGER)==LEDGER_SHA, "Frozen SHARP 72h ledger hash mismatch"
assert sha256(MASTER)==MASTER_SHA, "Frozen multimodal master hash mismatch"

print("Verified exact frozen inputs.")
print("SHARP:",SHARP)
print("SHARP ledger:",SHARP_LEDGER)
print("Master:",MASTER)
print("Output:",OUT)


In [ ]:
X=np.load(SHARP,mmap_mode="r",allow_pickle=False)
ledger=pd.read_csv(SHARP_LEDGER,low_memory=False)
master=pd.read_csv(MASTER,low_memory=False)

assert X.shape==(113433,3,16)
assert len(ledger)==113433
assert ledger["forecast_case_id"].is_unique
assert ledger["tensor_row"].is_unique
assert np.array_equal(np.sort(ledger["tensor_row"].to_numpy(int)),np.arange(len(X)))
assert ledger["label"].isin([-1,0,1]).all()
assert ledger["label_known"].astype(bool).equals(ledger["label"].ne(-1))
ledger["issue_dt"]=pd.to_datetime(ledger["issue_utc"],utc=True,errors="raise")
ledger["year"]=ledger["issue_dt"].dt.year.astype(int)

# The canonical tensor order is defined by tensor_row, not CSV row order.
ledger=ledger.sort_values("tensor_row",kind="stable").reset_index(drop=True)
assert np.array_equal(ledger["tensor_row"].to_numpy(int),np.arange(len(X)))

# Exact frozen 72h split/label support.
expected_roles={
 "train":25586,
 "model_validation":3905,
 "probability_calibration":2831,
 "conformal_calibration":4168,
 "policy_validation":13142,
 "retrospective_cycle25":35846,
 "supplementary_2026":11118,
 "unknown_label_excluded":16462,
 "purged_outcome_boundary":375,
}
actual=ledger["role"].value_counts().to_dict()
assert actual==expected_roles, f"Frozen role counts changed: {actual}"
assert int((ledger["role"].eq("train") & ledger["label_known"].astype(bool)).sum())==25586
assert int(ledger.loc[ledger["role"].eq("train"),"label"].sum())==1476

# ---------------------------
# Predeclared physical families
# ---------------------------
FAMILIES={
 "flux_pil":["USFLUX","R_VALUE"],
 "energy_shear":["TOTPOT","MEANSHR","SHRGT45"],
 "current_helicity":["TOTUSJZ","TOTUSJH","ABSNJZH","SAVNCPP","MEANJZD","MEANJZH","MEANALP"],
 "gradients":["MEANGBZ","MEANGBH","MEANGBT"],
 "inclination":["MEANGAM"],
}
SIGNED_MAG={"MEANJZD","MEANJZH","MEANALP"}
POSITIVE_MAG=set(EXPECTED_FEATURES)-SIGNED_MAG-{"MEANGAM"}
fidx={f:i for i,f in enumerate(EXPECTED_FEATURES)}

train_ref=ledger["role"].eq("train").to_numpy()
fit_mask=train_ref & ledger["label_known"].astype(bool).to_numpy() & ledger["label"].isin([0,1]).to_numpy()
assert fit_mask.sum()==25586

# ---------------------------
# Training-only robust physical transform
# ---------------------------
Z=np.empty_like(np.asarray(X),dtype=np.float64)
transform_rows=[]
for j,f in enumerate(EXPECTED_FEATURES):
    raw=np.asarray(X[:,:,j],dtype=float)

    if f in SIGNED_MAG:
        t=np.log1p(np.abs(raw))
        transform="log1p_abs"
    elif f in POSITIVE_MAG:
        ref_raw=raw[train_ref,:]
        if np.nanmin(ref_raw)<-1e-12:
            raise ValueError(f"Unexpected negative values for predeclared nonnegative magnitude {f}")
        t=np.log1p(np.maximum(raw,0.0))
        transform="log1p_nonnegative"
    elif f=="MEANGAM":
        t=raw.copy()
        transform="identity"
    else:
        raise AssertionError(f"Unclassified transform for {f}")

    ref=t[train_ref,:].reshape(-1)
    med=float(np.nanmedian(ref))
    q1,q3=np.nanpercentile(ref,[25,75])
    iqr=float(q3-q1)
    if not np.isfinite(iqr) or iqr<=1e-12:
        raise ValueError(f"Degenerate training IQR for {f}: {iqr}")

    Z[:,:,j]=np.clip((t-med)/iqr,-10,10)
    transform_rows.append({
        "feature":f,
        "physical_family":next(k for k,v in FAMILIES.items() if f in v),
        "transform":transform,
        "train_median":med,
        "train_iqr":iqr,
    })

pd.DataFrame(transform_rows).to_csv(OUT/"physical_feature_transform.csv",index=False)

# ---------------------------
# Magnetic Physical State Vector (MPSV)
# ---------------------------
family_slot={}
for fam,features in FAMILIES.items():
    jj=[fidx[f] for f in features]
    family_slot[fam]=np.nanmedian(Z[:,:,jj],axis=2)

# Three history slots are issue-288, -192, -96 native minutes.
slot_x=np.array([-2.,-1.,0.])
slot_c=slot_x-slot_x.mean()
denom=float((slot_c**2).sum())

mpsv_dict={}
for fam,A in family_slot.items():
    mpsv_dict[f"{fam}__latest"]=A[:,2]
    mpsv_dict[f"{fam}__net_change"]=A[:,2]-A[:,0]
    mpsv_dict[f"{fam}__slope_per96m"]=((A-A.mean(axis=1,keepdims=True))*slot_c[None,:]).sum(axis=1)/denom

phys=pd.DataFrame(mpsv_dict)
MPSV_COLS=phys.columns.tolist()
M=phys.to_numpy(float)
assert np.isfinite(M).all()

# ---------------------------
# Physical applicability: training-only Ledoit-Wolf Mahalanobis
# ---------------------------
lw=LedoitWolf().fit(M[train_ref])
delta=M-lw.location_
md2=np.einsum("ij,jk,ik->i",delta,lw.precision_,delta)
ood_q99=float(np.quantile(md2[train_ref],0.99))
phys["physical_md2"]=md2
phys["physical_ood_q99"]=md2>ood_q99

# ---------------------------
# Transparent physical reference model
# ---------------------------
y=ledger["label"].to_numpy(int)
physical_model=LogisticRegression(C=1.0,solver="lbfgs",max_iter=5000)
physical_model.fit(M[fit_mask],y[fit_mask])
phys["physical_probability"]=physical_model.predict_proba(M)[:,1]

coef=pd.DataFrame({
    "physical_axis":MPSV_COLS,
    "coefficient":physical_model.coef_[0],
})
coef.loc[len(coef)]={"physical_axis":"INTERCEPT","coefficient":float(physical_model.intercept_[0])}
coef.to_csv(OUT/"physical_logistic_coefficients.csv",index=False)

# ---------------------------
# Full 113,433-row physical ledger
# ---------------------------
full=ledger[["forecast_case_id","tensor_row","issue_utc","issue_dt","year","region_component_id",
             "role","label","label_known","probability_gru_mean","probability_logistic"]].copy()
for c in phys.columns:
    full[c]=phys[c].to_numpy()

def evidence_regime(role,year):
    if role=="retrospective_cycle25": return "cycle25_2021_2025_posthoc"
    if role=="supplementary_2026": return "supplementary_2026_posthoc"
    if role in {"train","model_validation","probability_calibration","conformal_calibration","policy_validation","purged_outcome_boundary"}:
        return "cycle24_earlier_development"
    if role=="unknown_label_excluded":
        return "unknown_label_excluded"
    return "other"

full["physical_regime"]=[evidence_regime(r,int(y)) for r,y in zip(full["role"],full["year"])]
full.to_csv(OUT/"physical_state_full_113433.csv.gz",index=False)

# ---------------------------
# Join to the frozen multimodal master for physical-vs-fusion consistency
# ---------------------------
master["forecast_case_id"]=master["forecast_case_id"].astype(str)
assert master["forecast_case_id"].is_unique
mm=master.merge(
    full[["forecast_case_id","probability_gru_mean",
          "physical_probability","physical_md2","physical_ood_q99","physical_regime"]+MPSV_COLS],
    on="forecast_case_id",how="inner",validate="one_to_one"
)
assert len(mm)==len(master)==71010

# The multimodal master carries the AIA probability; the frozen 113,433-row
# SHARP ledger is the authoritative source for the GRU probability.
mm["aia_probability"]=pd.to_numeric(mm["probability"],errors="coerce")
mm["sharp_probability"]=pd.to_numeric(mm["probability_gru_mean"],errors="coerce")
mm["fusion_probability"]=0.5*mm["aia_probability"]+0.5*mm["sharp_probability"]

eps=1e-6
def logodds(p):
    q=np.clip(np.asarray(p,float),eps,1-eps)
    return np.log(q/(1-q))

mm["physical_fusion_logit_gap"]=np.abs(
    logodds(mm["physical_probability"])-logodds(mm["fusion_probability"])
)

mm.to_csv(OUT/"physical_multimodal_case_table.csv.gz",index=False)

# ---------------------------
# H1 freeze receipt — no routing change
# ---------------------------
freeze={
 "status":"PHASE_H1_PHYSICAL_STATE_CONSTRUCTED",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "horizon_hours":72,
 "feature_order":EXPECTED_FEATURES,
 "physical_families":FAMILIES,
 "mpsv_columns":MPSV_COLS,
 "training_reference":{
   "role":"train",
   "cases":int(train_ref.sum()),
   "positives":int(y[fit_mask].sum()),
   "label_rule":"frozen 72h ledger label in {0,1}",
 },
 "physical_ood":{
   "estimator":"LedoitWolf",
   "score":"squared_mahalanobis",
   "q99_training_threshold":ood_q99,
 },
 "physical_model":"low_capacity_logistic_on_MPSV",
 "later_regime_evidence":"posthoc_already_seen_in_A_to_G",
 "source_sha256":{
   "sharp_tensor":sha256(SHARP),
   "sharp_72h_ledger":sha256(SHARP_LEDGER),
   "multimodal_master":sha256(MASTER),
 },
}
(OUT/"h1_freeze.json").write_text(json.dumps(freeze,indent=2)+"\n")

print("===== PHASE H1 PHYSICAL STATE CONSTRUCTION COMPLETE =====")
print(json.dumps(freeze,indent=2))
print("\nPhysical logistic coefficients:\n",coef.to_string(index=False))
print("\nOutputs:",OUT)


In [ ]:
# ============================================================
# H1 descriptive evaluation — NO policy selection
# ============================================================
known=full[full["label_known"].astype(bool) & full["label"].isin([0,1])].copy()

def metric_row(name,g):
    yy=g["label"].to_numpy(int)
    pp=g["physical_probability"].to_numpy(float)
    return {
      "group":name,
      "cases":int(len(g)),
      "positives":int(yy.sum()),
      "prevalence":float(yy.mean()) if len(yy) else np.nan,
      "physical_ap":float(average_precision_score(yy,pp)) if len(np.unique(yy))==2 else np.nan,
      "physical_auc":float(roc_auc_score(yy,pp)) if len(np.unique(yy))==2 else np.nan,
      "physical_brier":float(brier_score_loss(yy,pp)) if len(yy) else np.nan,
      "physical_ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_physical_md2":float(g["physical_md2"].median()),
    }

regime_rows=[]
for reg,g in known.groupby("physical_regime",sort=False):
    if reg=="unknown_label_excluded": continue
    regime_rows.append(metric_row(reg,g))
regime_summary=pd.DataFrame(regime_rows)
regime_summary.to_csv(OUT/"physical_regime_summary.csv",index=False)
print("\n=== Physical model/applicability by evidence regime ===")
print(regime_summary.to_string(index=False))

role_rows=[]
for role,g in known.groupby("role",sort=False):
    role_rows.append(metric_row(role,g))
role_summary=pd.DataFrame(role_rows)
role_summary.to_csv(OUT/"physical_role_summary.csv",index=False)

# Family-state shift: medians only, no outcome tuning.
family_shift=[]
for reg,g in full[full["physical_regime"].ne("unknown_label_excluded")].groupby("physical_regime",sort=False):
    row={"regime":reg,"cases":len(g)}
    for c in MPSV_COLS:
        row[f"median::{c}"]=float(g[c].median())
    family_shift.append(row)
family_shift=pd.DataFrame(family_shift)
family_shift.to_csv(OUT/"physical_family_shift_summary.csv",index=False)

# Year-by-year physical applicability to show whether shift is gradual or abrupt.
year_rows=[]
for yr,g in full.groupby("year",sort=True):
    year_rows.append({
      "year":int(yr),
      "cases":int(len(g)),
      "physical_ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_physical_md2":float(g["physical_md2"].median()),
    })
pd.DataFrame(year_rows).to_csv(OUT/"physical_applicability_by_year.csv",index=False)

# Physical-vs-fusion consistency and error association on the 71,010 multimodal rows.
label_col="label" if "label" in mm.columns else ("label_aia" if "label_aia" in mm.columns else None)
if label_col is None:
    raise ValueError("Cannot identify multimodal label column")
mm["label_final"]=pd.to_numeric(mm[label_col],errors="raise").astype(int)

risk=[]
for reg,g in mm.groupby("physical_regime",sort=False):
    z=g[np.isfinite(g["fusion_probability"]) & g["label_final"].isin([0,1])].copy()
    if not len(z): continue
    sqerr=(z["fusion_probability"].to_numpy(float)-z["label_final"].to_numpy(int))**2
    for metric in ["physical_md2","physical_fusion_logit_gap"]:
        x=z[metric].to_numpy(float)
        corr=float(pd.Series(x).corr(pd.Series(sqerr),method="spearman")) if np.nanstd(x)>0 else np.nan
        risk.append({
          "regime":reg,
          "metric":metric,
          "cases":int(len(z)),
          "positives":int(z["label_final"].sum()),
          "spearman_with_fusion_squared_error":corr,
        })
risk=pd.DataFrame(risk)
risk.to_csv(OUT/"physical_risk_association.csv",index=False)

print("\n=== Physical-state shift ===")
print(family_shift.to_string(index=False))
print("\n=== Physical risk association ===")
print(risk.to_string(index=False))

print("\nSTOP: H1 is descriptive. It does not alter NORMAL/DEGRADED/ABSTAIN routing.")
print("Review H1 outputs before defining H2 gates.")


## Stop point

Do **not** add the physical signal to routing from this notebook. First inspect the H1 coefficient signs, physical-axis distributions, later-regime OOD rates, consistency-gap behaviour and positive-class support. H2 must then predeclare any candidate physical gate using earlier development/policy support only. A null or harmful physical signal is a valid outcome and should remain diagnostic rather than being forced into the operational state machine.
